In [1]:
#This is the notebook where LP is using python + Claude to analyse data.
#The data source is the FEC Schedule A itemized receipts data via https://www.fec.gov/data/receipts/?data_type=processed&committee_id=C00915041&two_year_transaction_period=2026&is_individual=true

In [2]:
import pandas as pd

In [3]:
## Now let's make a copy of our lastest CSV file that is ONLY individual contributors.

In [4]:
dfAngieInd = pd.read_csv("LP_Indi_Angie_schedule_a-2026-08-18.csv")

In [5]:
dfAngieInd.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 78 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   committee_id                           252 non-null    str    
 1   committee_name                         252 non-null    str    
 2   report_year                            252 non-null    int64  
 3   report_type                            252 non-null    str    
 4   image_number                           252 non-null    int64  
 5   filing_form                            252 non-null    str    
 6   link_id                                252 non-null    int64  
 7   line_number                            252 non-null    str    
 8   transaction_id                         252 non-null    str    
 9   file_number                            252 non-null    int64  
 10  entity_type                            252 non-null    str    
 11  entity_type_desc 

In [6]:
###Claude Prompt: Write simple code block by block (to copy and paste) to tally all contributions, 
###tally all Florida contributions, and then show the percent if Florida contributions compared to other states.

In [7]:
##IMPORTANT NOTE FROM LP. This code tallies the total contributions from individuals only. It does NOT include PAC donations.
##And it adds together both negative values like refunds and positive values. 

In [8]:
## How much did this candidate recieve in total from individuals (not PACS)?

In [9]:
total_contributions = dfAngieInd['contribution_receipt_amount'].sum()
print(f"Total contributions: ${total_contributions:,.2f}")

Total contributions: $176,601.00


In [10]:
## How much did this candidate recieve in total from individuals (not PACS) in Florida?

In [12]:
florida_dfAngieInd = dfAngieInd[dfAngieInd['contributor_state'] == 'FL']
florida_totalInd = florida_dfAngieInd['contribution_receipt_amount'].sum()
print(f"Florida contributions: ${florida_totalInd:,.2f}")

Florida contributions: $167,781.00


In [13]:
## How much did this candidate recieve as a percentage from individuals (not PACS) in Florida versus other states?

In [14]:
other_states_total = total_contributions - florida_totalInd

florida_pct = (florida_totalInd / total_contributions) * 100
other_pct = (other_states_total / total_contributions) * 100

print(f"Florida: {florida_pct:.1f}% (${florida_totalInd:,.2f})")
print(f"Other states: {other_pct:.1f}% (${other_states_total:,.2f})")

Florida: 95.0% ($167,781.00)
Other states: 5.0% ($8,820.00)


In [15]:
## How much did this candidate recieve in total from individuals (not PACS) in all states?

In [16]:
state_totals = dfAngieInd.groupby('contributor_state')['contribution_receipt_amount'].sum().sort_values(ascending=False)
print(state_totals)

contributor_state
FL    167781.0
DC      1750.0
MD      1500.0
VA      1500.0
CA      1000.0
IL       750.0
GA       550.0
IN       500.0
KS       500.0
AZ       270.0
CO       250.0
OH       250.0
Name: contribution_receipt_amount, dtype: float64


In [17]:
## How much did this candidate recieve in total from individuals (not PACS) in all states. 
## How many individual TRANSACTIONS were there? (num_contributions)

In [18]:
##IMPORTANT NOTE: This counts individual contribution lines and not individual donors.

In [19]:
state_summary = dfAngieInd.groupby('contributor_state')['contribution_receipt_amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)
state_summary.columns = ['total_amount', 'num_contributions']
print(state_summary)

                   total_amount  num_contributions
contributor_state                                 
FL                     167781.0                234
DC                       1750.0                  3
MD                       1500.0                  2
VA                       1500.0                  3
CA                       1000.0                  1
IL                        750.0                  2
GA                        550.0                  2
IN                        500.0                  1
KS                        500.0                  1
AZ                        270.0                  1
CO                        250.0                  1
OH                        250.0                  1


In [20]:
###Claude prompt: Show each state as a percentage of total

In [21]:
## The total contributions from each state as a percentage. 

In [23]:
total_contributions = dfAngieInd['contribution_receipt_amount'].sum()

state_summary = dfAngieInd.groupby('contributor_state')['contribution_receipt_amount'].sum().sort_values(ascending=False)
state_summary_pct = (state_summary / total_contributions * 100).round(1)

for state, amount in state_summary.items():
    print(f"{state}: ${amount:,.2f} ({state_summary_pct[state]}%)")

FL: $167,781.00 (95.0%)
DC: $1,750.00 (1.0%)
MD: $1,500.00 (0.8%)
VA: $1,500.00 (0.8%)
CA: $1,000.00 (0.6%)
IL: $750.00 (0.4%)
GA: $550.00 (0.3%)
IN: $500.00 (0.3%)
KS: $500.00 (0.3%)
AZ: $270.00 (0.2%)
CO: $250.00 (0.1%)
OH: $250.00 (0.1%)
